<a href="https://colab.research.google.com/github/EberHernandezBenitez/EDP/blob/main/Bootstrap_y_Jackknife.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos Bootstrap y Jackknife.
Su objetivo principal es resolver un problema muy común en la ciencia de datos y la estadística: estimar la precisión de un estadístico (como la media, la mediana, la varianza o los coeficientes de una regresión) cuando la muestra de datos es pequeña, no conocemos la distribución matemática de fondo o las fórmulas matemáticas teóricas son demasiado complejas.

# El Método Bootstrap (Muestreo con reemplazo)
Si tenemos una muestra original de tamaño $n$, el Bootstrap genera miles de "réplicas" artificiales (por ejemplo, $B = 10,000$). Cada una de estas réplicas se construye seleccionando al azar $n$ elementos de tu muestra original, pero permitiendo el reemplazo (es decir, un mismo dato puede aparecer repetido varias veces en una réplica, mientras que otros pueden no aparecer).

Sirve para calcular el error estándar de un parámetro y también para construir intervalos de confianza altamente precisos sin importar si los datos son sesgados o asimétricos.

# 2. El Método Jackknife (Muestreo "deja-uno-fuera")

A diferencia del Bootstrap, el Jackknife no es aleatorio; es completamente determinista. Si tu muestra tiene un tamaño $n$, el Jackknife creará exactamente $n$ submuestras. Cada submuestra se construye eliminando rigurosamente un único dato diferente cada vez (un tamaño de $n-1$).Si tienes 12 datos, el Jackknife hará exactamente 12 evaluaciones: la primera sin el dato 1, la segunda sin el dato 2, y así sucesivamente.

Es excelente para reducir el sesgo de un estimador estadístico y sirve para identificar qué tan sensible es tu estimador ante la presencia de un dato atípico (outlier), midiendo cuánto cambia el resultado al removerlo.

**A continuación realizaremos un código que calcula ambos métodos para una muestra dada en clase.**

In [2]:
import random
import math

def calcular_media(datos):
    """Calcula la media aritmética de una lista."""
    return sum(datos) / len(datos)

def calcular_error_estandar_muestral(datos, media):
    """Calcula el error estándar muestral clásico como referencia."""
    n = len(datos)
    varianza = sum((x - media) ** 2 for x in datos) / (n - 1)
    return math.sqrt(varianza / n)

# 1. MÉTODO BOOTSTRAP
def remuestreo_bootstrap(datos, B=10000):
    """
    Aplica Bootstrap generando B muestras con reemplazo.
    Retorna la media de las medias y el error estándar Bootstrap.
    """
    n = len(datos)
    medias_bootstrap = []

    for _ in range(B):
        # Generar una muestra del mismo tamaño 'n' con reemplazo (muestreo repetido)
        muestra_b = [random.choice(datos) for _ in range(n)]
        medias_bootstrap.append(calcular_media(muestra_b))

    # Estimación final bajo Bootstrap
    media_bootstrap = calcular_media(medias_bootstrap)

    # Error estándar Bootstrap (desviación estándar de las medias replicadas)
    varianza_b = sum((m - media_bootstrap) ** 2 for m in medias_bootstrap) / (B - 1)
    error_estandar_b = math.sqrt(varianza_b)

    return media_bootstrap, error_estandar_b

# 2. MÉTODO JACKKNIFE
def remuestreo_jackknife(datos):
    """
    Aplica Jackknife eliminando un dato a la vez (deja uno fuera).
    Retorna la estimación Jackknife y su error estándar.
    """
    n = len(datos)
    medias_jackknife = []

    for i in range(n):
        # Crear submuestra omitiendo el elemento en la posición i
        muestra_j = datos[:i] + datos[i+1:]
        medias_jackknife.append(calcular_media(muestra_j))

    # Promedio de las estimaciones Jackknife
    media_jackknife_prom = calcular_media(medias_jackknife)

    # Error estándar Jackknife utilizando el factor multiplicador (n - 1) / n
    suma_cuadrados = sum((m - media_jackknife_prom) ** 2 for m in medias_jackknife)
    error_estandar_j = math.sqrt(((n - 1) / n) * suma_cuadrados)

    return media_jackknife_prom, error_estandar_j

# EJECUCIÓN DEL ANÁLISIS CON LA MUESTRA
if __name__ == "__main__":
    # Muestra de la clase:
    muestra = [35, 42, 38, 40, 45, 37, 39, 41, 44, 36, 43, 40]

    # Estadísticos muestrales estándar
    med_original = calcular_media(muestra)
    ee_original = calcular_error_estandar_muestral(muestra, med_original)

    # Aplicar técnicas de remuestreo
    med_boot, ee_boot = remuestreo_bootstrap(muestra, B=20000)
    med_jack, ee_jack = remuestreo_jackknife(muestra)

    # Resultados.
    print("ANALISIS DE RESULTADOS")
    print(f"Muestra: {muestra}")
    print(f"Tamaño de la muestra (n): {len(muestra)}")
    print("-" * 65)
    print(f"{'MÉTODO / TÉCNICA':<28}{'ESTIMACIÓN (MEDIA)':<22}{'ERROR ESTÁNDAR':<15}")
    print("-" * 65)
    print(f"{'Estadística Clásica':<28}{med_original:<22.4f}{ee_original:<15.4f}")
    print(f"{'Bootstrap (B=20,000)':<28}{med_boot:<22.4f}{ee_boot:<15.4f}")
    print(f"{'Jackknife (Deja-uno-fuera)':<28}{med_jack:<22.4f}{ee_jack:<15.4f}")

ANALISIS DE RESULTADOS
Muestra: [35, 42, 38, 40, 45, 37, 39, 41, 44, 36, 43, 40]
Tamaño de la muestra (n): 12
-----------------------------------------------------------------
MÉTODO / TÉCNICA            ESTIMACIÓN (MEDIA)    ERROR ESTÁNDAR 
-----------------------------------------------------------------
Estadística Clásica         40.0000               0.9129         
Bootstrap (B=20,000)        39.9957               0.8692         
Jackknife (Deja-uno-fuera)  40.0000               0.9129         


Como vemos, ambos son muy eficientes a comparación del cálculo con estadística clásica, tendriamos que compararlo con otras muestras de mayor complejidad para determinar cual es mejor o más eficiente.